In [2]:
import sys
from pathlib import Path

# Add parent directory to Python path
sys.path.insert(0, str(Path.cwd().parent))

from data.dataset import initialize_dataloaders

train_loader, val_loader, test_loader, ans2idx, idx2ans = initialize_dataloaders()

Train size: 79349
Val size:   51481
Test size:  37929


In [3]:
# import shutil
# from pathlib import Path

# VIT_EMB  = Path(r"D:\amir lab\RS-Vamba-main\image_extract\outputs\vit_encoded\embeddings")
# TEXT_EMB = Path(r"D:\amir lab\RS-Vamba-main\text_extract\outputs\text_encoded\embeddings")

# for p in VIT_EMB.glob("*.pt"):
#     p.unlink()
# for p in TEXT_EMB.glob("*.pt"):
#     p.unlink()

# print("Deleted. Free space now:")
# import shutil
# total, used, free = shutil.disk_usage(r"D:\\")
# print(f"  Free: {free/1e9:.1f} GB")

In [4]:
# import os
# import shutil
# from pathlib import Path

# # Define the directory
# CHECKPOINTS_DIR = Path("./outputs/vit_encoded/checkpoints")

# # Remove all .pt and .tmp files to start fresh
# if CHECKPOINTS_DIR.exists():
#     files_deleted = 0
#     for file in CHECKPOINTS_DIR.glob("*"):
#         if file.suffix in [".pt", ".tmp"]:
#             file.unlink()
#             files_deleted += 1
#     print(f"Successfully removed {files_deleted} old checkpoints. Directory is clean.")
# else:
#     print("Checkpoint directory does not exist yet. Nothing to remove.")

In [5]:
# list(sorted(Path("./outputs/vit_encoded/checkpoints").glob("epoch_*.pt")))

In [6]:
# import math
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from transformers import CLIPVisionModel
# from tqdm import tqdm
# from pathlib import Path
# import gc
# import os

# os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# VIT_DIR = Path("./outputs/vit_encoded")
# CHECKPOINTS_DIR = VIT_DIR / "checkpoints"
# EMBEDDINGS_DIR = VIT_DIR / "embeddings"
# for d in [VIT_DIR, CHECKPOINTS_DIR, EMBEDDINGS_DIR]: d.mkdir(parents=True, exist_ok=True)

# class CLIPViTEncoder(nn.Module):
#     def __init__(self, unfreeze_last=2):
#         super().__init__()
#         self.vit = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch16', use_safetensors=True)
#         for param in self.vit.parameters():
#             param.requires_grad = False
#         for layer in self.vit.encoder.layers[-unfreeze_last:]:
#             for param in layer.parameters():
#                 param.requires_grad = True
#         self.proj = nn.Sequential(
#             nn.Linear(768, 768),
#             nn.LayerNorm(768)
#         )

#     def forward(self, x):
#         if x.shape[-1] != 224:
#             x = F.interpolate(x, size=(224, 224), mode='bicubic', align_corners=False)
#         outputs = self.vit(pixel_values=x)
#         patch_tokens = outputs.last_hidden_state[:, 1:, :]
#         patch_tokens = self.proj(patch_tokens)
#         B, N, C = patch_tokens.shape
#         H = W = int(N ** 0.5)
#         return patch_tokens.transpose(1, 2).reshape(B, C, H, W)

# class ViTClassifier(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.encoder = CLIPViTEncoder()
#         self.head = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Flatten(),
#             nn.Linear(768, 512),
#             nn.LayerNorm(512),
#             nn.GELU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, num_classes)
#         )
#         for m in self.head.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.trunc_normal_(m.weight, std=0.02)
#                 if m.bias is not None:
#                     nn.init.constant_(m.bias, 0)

#     def forward(self, x):
#         features = self.encoder(x)
#         logits = self.head(features)
#         return logits, features

# def find_latest_checkpoint(ckpt_dir: Path):
#     checkpoints = sorted(ckpt_dir.glob("epoch_*.pt"))
#     return checkpoints[-1] if checkpoints else None

# gc.collect()
# torch.cuda.empty_cache()

# NUM_EPOCHS = 15
# WARMUP_EPOCHS = 3
# ACCUM_STEPS = 2
# device = torch.device('cuda')
# scaler = torch.amp.GradScaler('cuda')
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# model = ViTClassifier(num_classes=len(ans2idx)).to(device)

# optimizer = optim.AdamW([
#     {'params': model.encoder.parameters(), 'lr': 5e-6},
#     {'params': model.head.parameters(), 'lr': 2e-4}
# ], weight_decay=0.05, betas=(0.9, 0.999), eps=1e-8)

# steps_per_epoch = len(train_loader) // ACCUM_STEPS
# total_steps = NUM_EPOCHS * steps_per_epoch
# warmup_steps = WARMUP_EPOCHS * steps_per_epoch

# def lr_lambda(current_step):
#     if current_step < warmup_steps:
#         return current_step / max(1, warmup_steps)
#     progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
#     return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

# scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# latest_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR)
# START_EPOCH = 0
# if latest_ckpt:
#     ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
#     model.load_state_dict(ckpt['model_state_dict'])
#     optimizer.load_state_dict(ckpt['optimizer_state_dict'])
#     START_EPOCH = ckpt['epoch']

# best_val_loss = float('inf')

# for epoch in range(START_EPOCH, NUM_EPOCHS):
#     model.train()
#     epoch_loss = 0.0
#     pbar = tqdm(total=len(train_loader), desc=f"Epoch {epoch+1}")
#     optimizer.zero_grad(set_to_none=True)

#     for step, batch in enumerate(train_loader):
#         images = batch['image_pixel_values'].to(device, non_blocking=True)
#         labels = batch['answer_class_idx'].to(device, non_blocking=True)

#         with torch.amp.autocast('cuda'):
#             logits, _ = model(images)
#             loss = criterion(logits, labels) / ACCUM_STEPS

#         scaler.scale(loss).backward()

#         if (step + 1) % ACCUM_STEPS == 0:
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             scaler.step(optimizer)
#             scaler.update()
#             scheduler.step()
#             optimizer.zero_grad(set_to_none=True)

#         epoch_loss += loss.item() * ACCUM_STEPS
#         pbar.update(1)
#         if pbar.n % 10 == 0:
#             enc_lr = scheduler.get_last_lr()[0]
#             head_lr = scheduler.get_last_lr()[1]
#             pbar.set_postfix(loss=f"{loss.item() * ACCUM_STEPS:.4f}", enc_lr=f"{enc_lr:.1e}", head_lr=f"{head_lr:.1e}")

#     pbar.close()
#     avg_train_loss = epoch_loss / len(train_loader)

#     model.eval()
#     val_loss = 0.0
#     correct = 0
#     total = 0
#     with torch.no_grad():
#         for batch in val_loader:
#             imgs = batch['image_pixel_values'].to(device, non_blocking=True)
#             lbls = batch['answer_class_idx'].to(device, non_blocking=True)
#             with torch.amp.autocast('cuda'):
#                 lgt, _ = model(imgs)
#                 v_loss = criterion(lgt, lbls)
#             val_loss += v_loss.item()
#             preds = lgt.argmax(dim=-1)
#             correct += (preds == lbls).sum().item()
#             total += lbls.size(0)

#     avg_val_loss = val_loss / len(val_loader)
#     val_acc = correct / total

#     if avg_val_loss < best_val_loss:
#         best_val_loss = avg_val_loss
#         torch.save({
#             "epoch": epoch + 1,
#             "model_state_dict": model.state_dict(),
#             "optimizer_state_dict": optimizer.state_dict(),
#             "val_loss": avg_val_loss
#         }, CHECKPOINTS_DIR / "best_model.pt")

#     ckpt_path = CHECKPOINTS_DIR / f"epoch_{epoch+1:02d}_loss_{avg_val_loss:.4f}.pt"
#     torch.save({
#         "epoch": epoch + 1,
#         "model_state_dict": model.state_dict(),
#         "optimizer_state_dict": optimizer.state_dict(),
#         "val_loss": avg_val_loss
#     }, ckpt_path)

#     print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
#     gc.collect()
#     torch.cuda.empty_cache()



# @torch.no_grad()
# def extract_embeddings(loader, split_name, chunk_size=300):
#     model.eval()
#     chunk = {}
#     chunk_idx = 0

#     def save_chunk():
#         nonlocal chunk, chunk_idx
#         if chunk:
#             chunk_path = EMBEDDINGS_DIR / f"{split_name}_chunk_{chunk_idx:04d}.pt"
#             torch.save(chunk, chunk_path)
#             chunk = {}
#             chunk_idx += 1
#             gc.collect()

#     for batch in tqdm(loader, desc=f"Extracting {split_name}"):
#         images = batch['image_pixel_values'].to(device, non_blocking=True)
#         ids = batch['image_id']
#         with torch.amp.autocast('cuda'):
#             _, features = model(images)
#         features = features.cpu().to(torch.float16)
#         for i, img_id in enumerate(ids):
#             chunk[img_id] = features[i]
#         if len(chunk) >= chunk_size:
#             save_chunk()

#     save_chunk()
#     print(f"Saved {chunk_idx} chunks for {split_name} to {EMBEDDINGS_DIR}")

# for split_name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
#     save_path = EMBEDDINGS_DIR / f"{split_name}_chunk_0000.pt"
#     if save_path.exists():
#         print(f"Skipping {split_name}, already extracted.")
#     else:
#         extract_embeddings(loader, split_name)

# torch.save({
#     "model_state_dict": model.state_dict(),
#     "num_classes": len(ans2idx),
#     "ans2idx": ans2idx,
#     "idx2ans": idx2ans,
#     "embed_dim": 768,
#     "embedding_dir": str(EMBEDDINGS_DIR),
# }, VIT_DIR / "vit_final_model.pt")


In [7]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import CLIPVisionModel
from tqdm import tqdm
from pathlib import Path
import gc
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

VIT_DIR         = Path("./outputs/vit_encoded")
CHECKPOINTS_DIR = VIT_DIR / "checkpoints"
EMBEDDINGS_DIR  = VIT_DIR / "embeddings"
for d in [VIT_DIR, CHECKPOINTS_DIR, EMBEDDINGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


class CLIPViTEncoder(nn.Module):
    def __init__(self, unfreeze_last=6):
        super().__init__()
        self.vit = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch16', use_safetensors=True)
        for param in self.vit.parameters():
            param.requires_grad = False
        for layer in self.vit.encoder.layers[-unfreeze_last:]:
            for param in layer.parameters():
                param.requires_grad = True
        self.proj = nn.Sequential(
            nn.Linear(768, 768),
            nn.LayerNorm(768)
        )

    def forward(self, x):
        if x.shape[-1] != 224:
            x = F.interpolate(x, size=(224, 224), mode='bicubic', align_corners=False)
        outputs      = self.vit(pixel_values=x)
        patch_tokens = outputs.last_hidden_state[:, 1:, :]
        patch_tokens = self.proj(patch_tokens)
        B, N, C      = patch_tokens.shape
        H = W        = int(N ** 0.5)
        return patch_tokens.transpose(1, 2).reshape(B, C, H, W)


class ViTClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = CLIPViTEncoder()
        self.head    = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(768, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.encoder(x)
        logits   = self.head(features)
        return logits, features


def find_latest_checkpoint(ckpt_dir: Path):
    checkpoints = sorted(ckpt_dir.glob("epoch_*.pt"))
    return checkpoints[-1] if checkpoints else None


gc.collect()
torch.cuda.empty_cache()

NUM_EPOCHS  = 4
ACCUM_STEPS = 2
device      = torch.device('cuda')
scaler      = torch.amp.GradScaler('cuda')
criterion   = nn.CrossEntropyLoss()

model     = ViTClassifier(num_classes=len(ans2idx)).to(device)
optimizer = optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 2e-5},
    {'params': model.head.parameters(),    'lr': 5e-4}
], weight_decay=0.05, betas=(0.9, 0.999), eps=1e-8)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

latest_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR)
START_EPOCH = 0
if latest_ckpt:
    ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    START_EPOCH = ckpt['epoch']
    print(f"Resumed from epoch {START_EPOCH} | val_loss: {ckpt['val_loss']:.4f}")
else:
    print("No checkpoint found. Starting from scratch.")

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Total params:     {total:.1f}M")
print(f"Trainable params: {trainable:.1f}M")
print(f"Device:           {device}")
print(f"Training:         epoch {START_EPOCH + 1} → {NUM_EPOCHS}")

best_val_loss = float('inf')

for epoch in range(START_EPOCH, NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    pbar       = tqdm(total=len(train_loader), desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS}", leave=True)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        images = batch['image_pixel_values'].to(device, non_blocking=True)
        labels = batch['answer_class_idx'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits, _ = model(images)
            loss      = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        epoch_loss += loss.item() * ACCUM_STEPS
        pbar.update(1)
        if pbar.n % 10 == 0:
            pbar.set_postfix(loss=f"{loss.item() * ACCUM_STEPS:.4f}")

    pbar.close()
    scheduler.step()
    avg_train_loss = epoch_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    correct  = 0
    total_n  = 0
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['image_pixel_values'].to(device, non_blocking=True)
            lbls = batch['answer_class_idx'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                lgt, _ = model(imgs)
                v_loss = criterion(lgt, lbls)
            val_loss += v_loss.item()
            preds    = lgt.argmax(dim=-1)
            correct  += (preds == lbls).sum().item()
            total_n  += lbls.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc      = correct  / total_n

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss":             avg_val_loss,
        }, CHECKPOINTS_DIR / "best_model.pt")
        print(f"  New best model! val_loss: {avg_val_loss:.4f}")

    ckpt_path = CHECKPOINTS_DIR / f"epoch_{epoch+1:02d}_loss_{avg_val_loss:.4f}.pt"
    torch.save({
        "epoch":                epoch + 1,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss":             avg_val_loss,
    }, ckpt_path)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"train: {avg_train_loss:.4f} | "
          f"val: {avg_val_loss:.4f} | "
          f"acc: {val_acc:.4f} | "
          f"lr: {scheduler.get_last_lr()[0]:.2e}")

    gc.collect()
    torch.cuda.empty_cache()

print("Training complete.")


@torch.no_grad()
def extract_embeddings(loader, split_name, chunk_size=300):
    model.eval()
    chunk     = {}
    chunk_idx = 0

    def save_chunk():
        nonlocal chunk, chunk_idx
        if chunk:
            chunk_path = EMBEDDINGS_DIR / f"{split_name}_chunk_{chunk_idx:04d}.pt"
            torch.save(chunk, chunk_path)
            chunk      = {}
            chunk_idx += 1
            gc.collect()

    for batch in tqdm(loader, desc=f"Extracting {split_name}"):
        images = batch['image_pixel_values'].to(device, non_blocking=True)
        ids    = batch['image_id']
        with torch.amp.autocast('cuda'):
            _, features = model(images)
        # (B, 768, 14, 14) → (B, 196, 768) float16
        features = features.flatten(2).transpose(1, 2).cpu().to(torch.float16)
        for i, img_id in enumerate(ids):
            chunk[img_id] = features[i]
        if len(chunk) >= chunk_size:
            save_chunk()

    save_chunk()
    print(f"Saved {chunk_idx} chunks for {split_name} to {EMBEDDINGS_DIR}")


for split_name, loader in [
    ("train", train_loader),
    ("val",   val_loader),
    ("test",  test_loader)
]:
    first_chunk = EMBEDDINGS_DIR / f"{split_name}_chunk_0000.pt"
    if first_chunk.exists():
        print(f"Skipping {split_name} — already extracted")
    else:
        extract_embeddings(loader, split_name)

torch.save({
    "model_state_dict": model.state_dict(),
    "num_classes":      len(ans2idx),
    "ans2idx":          ans2idx,
    "idx2ans":          idx2ans,
    "embed_dim":        768,
    "embedding_dir":    str(EMBEDDINGS_DIR),
}, VIT_DIR / "vit_final_model.pt")

print("Final model saved →", VIT_DIR / "vit_final_model.pt")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7239.34it/s]
[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight    

No checkpoint found. Starting from scratch.
Total params:     86.9M
Trainable params: 43.6M
Device:           cuda
Training:         epoch 1 → 4


Epoch 01/4: 100%|██████████| 2480/2480 [13:51<00:00,  2.98it/s, loss=3.1908]


  New best model! val_loss: 2.4939
Epoch 01/4 | train: 2.3043 | val: 2.4939 | acc: 0.2867 | lr: 1.71e-05


Epoch 02/4: 100%|██████████| 2480/2480 [14:11<00:00,  2.91it/s, loss=2.3549]


...

Saved 240 chunks for train to outputs\vit_encoded\embeddings


Extracting val: 100%|██████████| 1609/1609 [09:11<00:00,  2.92it/s]


Saved 147 chunks for val to outputs\vit_encoded\embeddings


Extracting test: 100%|██████████| 1186/1186 [07:06<00:00,  2.78it/s]


Saved 109 chunks for test to outputs\vit_encoded\embeddings
Final model saved → outputs\vit_encoded\vit_final_model.pt


In [8]:
import torch
from pathlib import Path

EMBEDDINGS_DIR = Path("./outputs/vit_encoded/embeddings")

for split in ["train", "val", "test"]:
    files = sorted(EMBEDDINGS_DIR.glob(f"{split}*.pt"))
    total_items = 0
    shape = None
    for f in files:
        data = torch.load(f, map_location='cpu', weights_only=False)
        total_items += len(data)
        if shape is None:
            shape = data[list(data.keys())[0]].shape
        del data
    print(f"{split}: {len(files)} file(s) | {total_items} images | shape: {shape}")

train: 240 file(s) | 74371 images | shape: torch.Size([196, 768])
val: 147 file(s) | 46379 images | shape: torch.Size([196, 768])
test: 109 file(s) | 34415 images | shape: torch.Size([196, 768])


In [1]:
from pathlib import Path
import torch

VIT_DIR  = Path(r"D:\amir lab\RS-Vamba-main\image_extract\outputs\vit_encoded\embeddings")
TEXT_DIR = Path(r"D:\amir lab\RS-Vamba-main\text_extract\outputs\text_encoded\embeddings")

for name, emb_dir, pattern in [
    ("ViT",     VIT_DIR,  "{split}*.pt"),
    ("RoBERTa", TEXT_DIR, "{split}_chunk_*.pt"),
]:
    print(f"\n{name}:")
    for split in ["train", "val", "test"]:
        files = sorted(emb_dir.glob(pattern.format(split=split)))
        if not files:
            print(f"  {split}: no files found")
            continue
        size_mb = sum(f.stat().st_size for f in files) / 1e6
        sample_data = torch.load(files[0], map_location='cpu', weights_only=False)
        first_val = list(sample_data.values())[0]
        if isinstance(first_val, dict):
            shape = {k: v.shape for k, v in first_val.items() if hasattr(v, 'shape')}
        elif hasattr(first_val, 'shape'):
            shape = first_val.shape
        else:
            shape = type(first_val)
        items_in_first = len(sample_data)
        del sample_data
        print(f"  {split}: {len(files)} files | ~{len(files)*items_in_first} items | {size_mb:.0f} MB | shape: {shape}")


ViT:
  train: 240 files | ~74160 items | 23896 MB | shape: torch.Size([196, 768])
  val: 147 files | ~45423 items | 15503 MB | shape: torch.Size([196, 768])
  test: 109 files | ~33681 items | 11422 MB | shape: torch.Size([196, 768])

RoBERTa:
  train: 78 files | ~79872 items | 7941 MB | shape: {'cls_embed': torch.Size([768]), 'token_embed': torch.Size([11, 768])}
  val: 51 files | ~52224 items | 5152 MB | shape: {'cls_embed': torch.Size([768]), 'token_embed': torch.Size([11, 768])}
  test: 38 files | ~38912 items | 3795 MB | shape: {'cls_embed': torch.Size([768]), 'token_embed': torch.Size([11, 768])}


In [3]:
from pathlib import Path

ckpt_dir = Path(r"d:\amir lab\RS-Vamba-main\fusion\outputs\checkpoints")

# keep only best_model.pt, delete all epoch_*.pt
deleted = 0
for f in ckpt_dir.glob("epoch_*.pt"):
    f.unlink()
    print(f"Deleted: {f.name}")
    deleted += 1

print(f"\nDone. Deleted {deleted} checkpoints.")
print("Remaining files:")
for f in sorted(ckpt_dir.glob("*.pt")):
    print(f"  {f.name}")

Deleted: epoch_01_acc_0.7490.pt
Deleted: epoch_02_acc_0.7390.pt

Done. Deleted 2 checkpoints.
Remaining files:


In [4]:
best = ckpt_dir / "best_model.pt"
if best.exists():
    best.unlink()
    print("Deleted best_model.pt")